# 1. Import Libraries

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import pipeline
import os

!pip install deep_translator
from deep_translator import GoogleTranslator

# 2. Get Model

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Create Prompt Function

In [ ]:
def prompt_llm(prompt: str):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        repetition_penalty=1.5,
        no_repeat_ngram_size=4
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


## 3.1 Test Prompt

In [ ]:

prompt = """
Paraphrase the following sentence without changing its positive sentiment:
"Film ini sangat menarik dan akting para pemainnya luar biasa."
"""

prompt_llm(prompt)

# 4. Count Dataset Sentiments

In [ ]:
import pandas as pd

df = pd.read_csv("../../dataset/dataset.csv")

sentiment_counts = {}

for index, row in df.iterrows():
    sentiment = row["manual sentiment"]
    
    if (sentiment not in sentiment_counts):
        sentiment_counts[sentiment] = 1
    else:
        sentiment_counts[sentiment] += 1

highest_sentiment_tuple = ("", 0)

for key in sentiment_counts.keys():
    value = sentiment_counts[key]
    if (value > highest_sentiment_tuple[1]):
        highest_sentiment_tuple = (key, value)

sentiment_generation_counts = {}

for key in sentiment_counts.keys():
    value = sentiment_counts[key]
    _, highest_sentiment_count = highest_sentiment_tuple

    sentiment_generation_counts[key] = highest_sentiment_count -  value

print("The LLM needs to generate more data according to these counts: ")
print(sentiment_generation_counts)

# 5. Generate Prompts

In [ ]:
def create_prompt(sentiment: str) -> str:

    examples = {
        "Positive": """
Donald Trump's new tariff policy created optimism among Indonesian exporters.
Several trade analysts said the policy could improve export growth and investment opportunities.
Business groups welcomed the stronger economic relationship between both countries.
""",

        "Negative": """
Donald Trump's tariff policy increased pressure on Indonesian export industries.
Economists warned that higher trade barriers could weaken trade performance and reduce profits.
Several companies expressed concern about rising uncertainty in global markets.
""",

        "Neutral": """
Donald Trump announced a new tariff policy affecting international trade.
Indonesian officials are reviewing the potential impact on export activities.
Economists said the long-term economic effects are still unclear.
"""
    }

    prompt = f"""
Write a {sentiment.lower()} economic news article in English about Donald Trump's tariff policies and Indonesia.

Use this example as guidance:

{examples[sentiment]}

Write exactly 3 short sentences.
Write in formal news style.
Do not copy the example.
Article:
"""

    return prompt

prompts_per_sentiment = {}

for key in sentiment_counts.keys():
    prompts_per_sentiment[key] = create_prompt(key)

prompts_per_sentiment

# 6. Create Translation Pipeline

In [ ]:
def translate_to_indonesian(text: str):
    return GoogleTranslator(
        source="en",
        target="id"
    ).translate(text)

## 6.1 Test Run

In [ ]:

for key in prompts_per_sentiment.keys():
    result = prompt_llm(prompts_per_sentiment[key])
    translated = translate_to_indonesian(result)
    print(key, ": ", result)
    print("Translated: " + translated)

# 7. Generate New Data into Dataset

In [ ]:
file_path = "../../outputs/RAG/rag_results.csv"

if os.path.exists(file_path):
    generated_df = pd.read_csv(file_path)
else:
    generated_df = pd.DataFrame(columns=["sentiment", "text_en", "text_id", "llm_generated"])

existing_counts = generated_df["sentiment"].value_counts().to_dict()

for sentiment in sentiment_generation_counts.keys():
    amount = sentiment_generation_counts[sentiment] - existing_counts[sentiment]

    for i in range(amount):
        while True:
            try:
                result = prompt_llm(prompts_per_sentiment[sentiment])
                translated = translate_to_indonesian(result)
            
                generated_df.loc[len(generated_df)] = {
                "sentiment": sentiment,
                "text_en": result,
                "text_id": translated,
                "llm_generated": True
            }

                print(sentiment, i + 1, "of", amount)
                print("  Result:", result)
                print("  Translated: ", translated)

                generated_df.to_csv(file_path, index=False)
                break
            except Exception as e:
                print(i, e)

generated_df